# Removing Subjects for both Discrete and Continuous ratings

We need to 
* plot all of the discretized ratings again, in a form which allows us to see which should be removed.
* Agree on a method to remove subjects with low pain fluctuation (because such correlations are likely to be spurious, as in the subject didn't know they were changing their rating). Rating changes smaller than the minimum felt distance on the VAS scale are not typically felt by patients to be a different pain rating.
* Bad data is - someone who isn't moving the slider enough (maximum change across entire time series is below minimum felt score)
* Bad data is also - someone who's pain ratings were very low. This is because sensitivity to pain is decreases with very low pain ratings, so these are likely to be unreliable recordings.

In [2]:
from Carl_Response_Functions.load_responses import readCsv
import numpy as np
import math

import matplotlib.pyplot as plt
path = f'Results/Discretized_responses/Maeghal_preprocessed_responses'

In [3]:
def check_responses(path, low_range=10, felt_score=10, smooth=True,plot=False):
    """If all the values in the array are below the low_range then make the reject_flag = True, 
    calculate diff(-1) of the array and if no values of the difference between the 
    previous and current value turns out to be greater than the felt_score then reject_flag = True
    """
    
    if smooth == True:
        response_matrix = readCsv("responseArray.csv", path=path)
    else:
        response_matrix = readCsv("unsmoothedResponseArray.csv", path=path)
    print(response_matrix.shape)
    reject_flags = np.zeros(response_matrix.shape[0])
    for index in np.arange(0, response_matrix.shape[0], 1):
        response_array = response_matrix.T[:, index]
        
        if np.all(response_array < low_range):
            # print("all values below: ", low_range, "for: ", index)
            reject_flags[index] = 1

        # differences = np.diff(response_array)
        # if not np.any(differences > felt_score):
        #     # print("all differences below: ", felt_score, "for: ", index)
        #     reject_flags[index] = 2
        differences = np.max(response_array) - np.min(response_array)
        if not (differences >= felt_score):
            # print("all differences below: ", felt_score, "for: ", index)
            reject_flags[index] = 2
    
    percentage_of_rejects = ((np.sum(reject_flags == 1) + np.sum(reject_flags == 2)) / len(reject_flags)) * 100
    print("Data retained: {:.2f}% Data rejected: {:.2f}%".format(100 - percentage_of_rejects,percentage_of_rejects,))
    
    colors = ['red', 'blue', 'green', 
              'orange', 'brown', 'purple',
              'black', 'grey', 'pink', 'blueviolet',
              'darkslategrey', 'darkturquoise', 'darkviolet',
              'deeppink', 'deepskyblue', 'dimgray',
              'dimgrey']
    
    response_category_and_flag = {0: "Not rejected", 1: "Rejected all data below threshold", 2: "Rejected all diff below felt score"}
    num_cols = 4
    if plot:
        for flag in response_category_and_flag.keys():
            # fig, axes = plt.subplots(num_rows, num_cols, figsize=(16, 4 * num_rows))
            num_rows = math.ceil(np.sum(reject_flags == flag) / num_cols)
            try:
                fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(16, 2 * num_rows))
                fig.suptitle(response_category_and_flag[flag], fontsize=24)
                
                axes = axes.flatten()
                subplot_idx = 0
                for index in np.arange(0, response_matrix.shape[0], 1):
                    response_array = response_matrix.T[:, index]
                    color = colors[index % 16]

                    if reject_flags[index] == flag:
                        axes[subplot_idx].plot(response_array, color=color)
                        axes[subplot_idx].set_title(f"Index {index}")

                        subplot_idx += 1
                # # Hide empty subplots
                # for i in range(subplot_idx, num_cols * num_rows + 1):
                #     row_idx = (i - 1) // num_cols
                #     col_idx = (i - 1) % num_cols
                #     fig.delaxes(axes[row_idx, col_idx])
                plt.tight_layout()
                plt.subplots_adjust(top=0.97)
                #plt.savefig(f"{response_category_and_flag[flag]}.pdf")
                plt.show()
                
            except Exception as e:
                print(e)
    
    return reject_flags

In [4]:
reject_flags = check_responses(path=path,smooth=True, plot=False)

(390, 240)
Data retained: 82.05% Data rejected: 17.95%
